# 1. Extraire les données des destinations
* Objectif : Obtenir les coordonnées GPS des 35 villes
* Comment : API Nominatim API
* Données à extraire : CSV (ID_ville, ville, latitude, longitude)

(bibliothèque python également disponible)

In [ ]:
#liste des 35 villes
villes =["Mont Saint Michel",
"St Malo",
"Bayeux",
]


In [ ]:
# Importer les bibliothèques nécessaires
import requests # pour faire des requêtes HTTP
import time # pour ajouter un délai entre les requêtes
import uuid # pour générer des identifiants uniques
import pandas as pd # pour manipuler les données
import csv # pour lire et écrire des fichiers CSV

In [78]:
# Récupérer les coordonnées de chaque ville
from villes_coordonnees_meteo_hotels import coordonnees_villes


 Clé API chargée avec succès : 29e9493ff0392fd7df6e77036ab94b5a
Météo ajoutée pour Mont Saint Michel
Météo ajoutée pour Saint Malo
Nombre de villes traitées : 2
Fichier 'villes_meteo.csv' exporté avec les coordonnées et la météo.


# 2. Collecte des données météo (prévisions +7 jours)
* Objectif : Obtenir les prévisions météo des 35 villes
* Comment : 
    * OpenWeatherMap - One Call API 
    * Critères : températures, pluie, humidité, qualité de l'air
* Sortie : 

In [79]:
from villes_coordonnees_meteo_hotels import meteo_villes

# 3. Scraping Booking.com
* Quoi : récupérer les informations d'hôtels pour chaque ville
* Comment : scrapy
* Données à extraire : CSV (ID_hôtel, city_ID, nom_hôtel, url, latitude, longitude, note, description)

In [80]:
from villes_coordonnees_meteo_hotels import scrape_booking


# 4. Création du Data Lake (S3)
* Quoi : Stocker tous les fichiers CSV dans des buckets S3
* Contenu à stocker : Données météo par ville + Liste des villes avec GPS / Données des hôtels
* Nom du bucket S3 : kayak-data-lake

In [82]:
import load_dotenv
import os
import boto3
from botocore.exceptions import NoCredentialsError, ClientError

# Charger les variables d'environnement depuis le fichier .env
load_dotenv() 
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")
aws_region = os.getenv("AWS_REGION")


def upload_file_to_s3(local_file_path, bucket_name, s3_file_key):
    s3 = boto3.client('s3',
                    aws_access_key_id=aws_access_key_id,
                    aws_secret_access_key=aws_secret_access_key,
                    region_name=aws_region)
    try:
        s3.upload_file(local_file_path, bucket_name, s3_file_key)
        print(f" Fichier {local_file_path} uploadé vers s3://{bucket_name}/{s3_file_key}")
    except FileNotFoundError:
        print("Le fichier local n'a pas été trouvé.")
    except NoCredentialsError:
        print("Les identifiants AWS ne sont pas configurés.")
    except ClientError as e:
        print(f"Erreur lors de l'upload vers S3 : {e}")

# Exemple d'utilisation juste après avoir sauvegardé le CSV localement
chemin_fichier = "D:/Profils/NLefort/Desktop/JEDHA/PROJETS/03. Data_collection_and_management"
local_csv_path = chemin_fichier + "/all_hotels.csv"
bucket_name = "nom-de-ton-bucket-s3"
s3_key = "dossier_sur_s3/all_hotels.csv"  # chemin/fichier dans le bucket

upload_file_to_s3(local_csv_path, bucket_name, s3_key)


ModuleNotFoundError: No module named 'load_dotenv'

# 5. ETL vers un entrepôt SQL
* Outils : AWS RDS (MySQL ou PostgreSQL)
* Étapes : Créer des tables SQL pour villes, meteo, hotels
* Charger les fichiers CSV depuis S3 dans la base de données avec un script ETL (Python, Airflow ou AWS Glue)
* Vérifier que les données sont bien normalisées (clé étrangère entre weather et cities, entre hotels et cities)

# 6. Visualisations
Outil recommandé : Plotly
Cartes à produire :
* Top 5 des villes avec la meilleure météo (selon ton indice)
* Top 20 hôtels (note utilisateur + météo favorable)

Représentation : cartes interactives avec clusters ou bulles



# Livrables finaux
* CSV enrichi → Stocké sur S3
* Base SQL sur AWS RDS contenant toutes les données
* Deux cartes Plotly :
    * Top 5 des destinations
    * Top 20 hôtels
* (Optionnel) : Documentation sur les critères météo utilisés + scripts utilisés